# 03 · ADS-B de adsb.lol

Las trazas salen de dos sitios:

1. 2025 y enero de 2026 (también febrero en modo final): el ClickHouse público de ADSB.exposed
   (`scripts/descarga_adsb_exposed.py`), una consulta por aeropuerto y bloque de 4 días. Solo enviamos fecha, hora y
   recuadro. Trae adsb.lol y airplanes.live, pero solo usamos adsb.lol.
2. Julio de 2026 (también junio en modo final): el archivo `adsblol/globe_history_2026` de GitHub
   (`scripts/descarga_adsb_github.py`), unos 4 GB por día procesados en streaming, porque desde mayo de 2026 adsb.lol
   ya no está en ADSB.exposed.

`scripts/variables_adsb.py` detecta el despegue en la traza, la enlaza con la salida por callsign y hora (±240 s) y
mide el último punto quieto, el primer movimiento, la primera aparición, las paradas, la cola de los últimos 10 min,
el recorrido, la posición inicial, los huecos de cobertura y las diferencias con la referencia NM/P10.
`scripts/variables_stand.py` sitúa cada stand con los aviones vistos aparcados y mide la distancia al primer punto de
la traza.

Por defecto reutilizamos las variables con las que se entrenó v30.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT)]
from cuaderno import elegir_periodo, existen, interruptor, run  # noqa: E402

PERIODO = elegir_periodo("ranking")   # abierto suelto: "ranking" (ene + jul 2026) o "final" (ene, feb, jun y jul 2026)
FINAL = PERIODO == "final"
import periodo as PER  # noqa: E402  rutas del periodo: PER.RANKING (fichero a predecir), PER.FEAT (variables)

DESCARGAR = interruptor("DESCARGAR_ADSB", False)
REHACER = interruptor("REHACER_ADSB", False)

In [ ]:
if DESCARGAR:
    run("scripts/descarga_adsb_exposed.py", "2025-01-01", "2025-12-31", "--hilos", "3", "--bloque", "4")
    run("scripts/descarga_adsb_exposed.py", "2026-01-01", "2026-02-28" if FINAL else "2026-01-31", "--hilos", "3", "--bloque", "4")
    if FINAL:
        run("scripts/descarga_adsb_github.py", "2026-06-01", "2026-06-30", "--procesos", "6")
    run("scripts/descarga_adsb_github.py", "2026-07-01", "2026-07-31", "--procesos", "6")
else:
    print("Trazas ADS-B: no se descargan (DESCARGAR_ADSB = 0)")

In [ ]:
if FINAL:   # adsb_2025 se copia de Data/features; 2026 y las posiciones de los stands se rehacen con los cuatro meses
    run("scripts/preparar_final.py", "adsb", "stand", *(["--rehacer"] if REHACER else []))
else:
    if REHACER or not existen("Data/features/adsb_2025.parquet", "Data/features/adsb_2026.parquet"):
        run("scripts/variables_adsb.py", "2025", "10")
        run("scripts/variables_adsb.py", "2026", "10")
    else:
        print("Variables ADS-B: se reutilizan Data/features/adsb_{2025,2026}.parquet")
    if REHACER or not existen(*(f"Data/features/adsb_stand{s}_{a}.parquet" for s in ("", "60") for a in (2025, 2026))):
        run("scripts/variables_stand.py")
    else:
        print("Puerta/stand: se reutilizan Data/features/adsb_stand*_{2025,2026}.parquet")

Cobertura: porcentaje de salidas con traza enlazada, por aeropuerto.

In [ ]:
import duckdb
import pandas as pd

raw = (ROOT / "Data" / "raw").as_posix()
filas = {}
for anio, src in ((2025, f"{raw}/training_*.parquet"), (2026, PER.RANKING.as_posix())):
    dep = duckdb.sql(f"select MVT_ID_mvt, ADEP_mvt airport from '{src}' where PHASE_mvt = 'DEP'").df()
    a = pd.read_parquet(PER.FEAT / f"adsb_{anio}.parquet", columns=["MVT_ID_mvt"])
    filas[anio] = dep.assign(enlazada=dep.MVT_ID_mvt.isin(a.MVT_ID_mvt)).groupby("airport").enlazada.mean().mul(100)
pd.DataFrame(filas).round(1)